# Loop 1 · Manual: the agent skips the lookup

The Acme Cloud support agent runs on **OpenClaw**. Its behavior is one file, `AGENTS.md`, which the
harness keeps in the MLflow Prompt Registry and renders into OpenClaw's workspace for every call.
The policies it should answer from sit next to it in `knowledge/`. The **MLflow connector**
(`connectors/openclaw-mlflow`) logs every OpenClaw run as its own trace, with an `LLM` span for
its model calls and a `TOOL` span per tool call.

Four loops, each finding one failure and fixing it with measurable evidence. v1 ships with the
first two; the other two arrive later, when other teams edit AGENTS.md and ship it without the
gate. Each loop hands more of the work to the harness:

| Loop | Failure | Signal | Store | Update | Promotes |
|---|---|---|---|---|---|
| 1 | Skips the lookup | you | you | you | you |
| 2 | Makes things up | you + a grounding judge | triage + approve | the analyst | you |
| 3 | Unsafe account help, from another team's edit | you + an aligned judge, must-pass check | triage + approve | an optimizer | you |
| 4 | Bad escalation, from another team's edit | a 50% label sample + scheduled scoring | `cycle` | `cycle` | the gate |

In this loop a person does every step. All results come from `session/store/` (the precomputed
run); `RUNBOOK.md` has the same steps as `selfimprove` commands. Cells marked **Under the hood**
say which MLflow calls produced each step's evidence, and read it back with the raw MLflow API.

In [ ]:
import _session as s

experiment_id = s.setup(1)
r = s.results(1)

## 1. Find: what v1 tells the agent

Look at the first bullet and the quick facts. Then compare them with `knowledge/`: refunds are
14 days, reset links last 15 minutes, lockouts 30 minutes, and Pro has 10 seats.

In [ ]:
from selfimprove import instructions

print(instructions.load(1).template)

**Under the hood:** `instructions.load(1)` is `mlflow.genai.load_prompt("prompts:/acme-agents-md/1")`. The Prompt
Registry keeps every version of AGENTS.md with a commit message and tags, and the `production`
alias says which one the agent runs. Before each call the harness loads the version under test and
writes it into a fresh OpenClaw workspace as `AGENTS.md`. Shipping a change means moving that
alias; nothing is redeployed. At the end of the precomputed run, `production` points at:

In [ ]:
import mlflow
from selfimprove import config

name = config.settings.prompt_name
p = mlflow.genai.load_prompt(f"prompts:/{name}@production")
print(f"{p.name} @production -> v{p.version}: {p.commit_message}")

## 2. Signal: production traffic, as the reviewers saw it

Eight customer questions went to production on v1. Each one is a trace. The reviewers (you, in
the room) labeled each answer pass or fail and attached the correct answer to every failure.

In [ ]:
before = s.loop_traces(1, "before")
s.show(s.answers(before))

**Under the hood:** each question went through `agents.run()`, which opens an `AGENT` span with `mlflow.start_span(...)`
and tags the trace with `mlflow.update_current_trace(tags={"si_source": "production",
"si_prompt_version": ...})`. Every later step finds real traffic by that tag
(`tags.si_source = 'production'`) and keeps evaluation runs out of it (`si_source = 'eval'`). The
session also tagged each trace with its loop and phase, which is what `s.loop_traces` filters on:

In [ ]:
t = before[0]
print(t.info.trace_id, t.info.state)
{k: v for k, v in t.info.tags.items() if not k.startswith("mlflow.")}

Why did it fail? The harness trace only says "OpenClaw answered". The connector's trace for the
same run shows what OpenClaw actually did: one model call and **no `read`**. The harness links
the two with the run's session key (`openclaw.session_key` → `mlflow.trace.session`).

In [ ]:
s.tree(before[0])

**Under the hood:** the harness span records `openclaw.session_key`, and the connector writes the same key into its own
trace's metadata as `mlflow.trace.session`. One `mlflow.search_traces` call on that metadata finds
OpenClaw's trace for this run:

In [ ]:
key = next(sp.attributes["openclaw.session_key"] for sp in before[0].data.spans
           if "openclaw.session_key" in sp.attributes)
oc = mlflow.search_traces(locations=[experiment_id], return_type="list",
                          filter_string=f"metadata.`mlflow.trace.session` = '{key}'")[0]
print(key)
[(sp.name, sp.span_type) for sp in oc.data.spans]

That observation is cheap to turn into code. Loop 1's verifier (`session/criteria/loop1.yaml`) adds
two checks:

* `read_knowledge`: the connector's trace must have a successful `read` of a `knowledge/` file
* `no_stale_facts`: **must-pass**, no quoted stale fact (30-day, 24 hours, 25 seats, 1 hour)

`selfimprove score` ran them, and the `quality` judge, on every production trace:

In [ ]:
s.show(s.verdicts(before))

In [ ]:
s.show(s.failure_rate(before).to_frame("failure rate"), pct=True)

**Under the hood:** every verdict is an **assessment** attached to the trace, with a source. A reviewer's label is
`mlflow.log_feedback(name="quality", ..., source=HUMAN)`, plus `mlflow.log_expectation(name=
"expected_response")` for the correct answer. `selfimprove score` ran `mlflow.genai.evaluate(data=
<these traces>, scorers=[...])` with **no** `predict_fn`, so nothing re-ran: the checks and judges
read the recorded trace and attached `CODE` and `LLM_JUDGE` feedback next to the human label.

In [ ]:
for a in before[0].info.assessments:
    print(f"{a.name:<22} {str(a.value)[:14]:<14} {a.source.source_type:<12} {a.source.source_id}")

## 3. Store: the failures become tests

Each labeled failure becomes a dataset row: the question as input, the reviewer's correct answer
as the expectation, and its provenance as tags. A hash of the question puts about one in five in
the **holdout**, which only the gate ever sees. The eight golden holdout cases were there from
the start (`session/golden.yaml`).

In [ ]:
from selfimprove import store

print(r["dataset"])
df = store.dataset().to_df()
df = df[df["tags"].map(lambda t: t.get("source") == "golden" or t.get("author") == "you")]  # as of loop 1
s.show(df.assign(question=df["inputs"].map(lambda i: i["question"][:70]),
                 split=df["tags"].map(lambda t: t.get("split")),
                 source=df["tags"].map(lambda t: t.get("source")),
                 author=df["tags"].map(lambda t: t.get("author")))[["question", "split", "source", "author"]])

**Under the hood:** the dataset is an MLflow evaluation dataset (`mlflow.genai.datasets.create_dataset`), and new tests
arrive with `dataset.merge_records([...])`. Each record is `{inputs, expectations, tags}`; the tags
carry the provenance (`source`, `author`, `source_trace`) and the `split`. The gate and the
regression suite read it back with `get_dataset`:

In [ ]:
from mlflow.genai.datasets import get_dataset

ds = get_dataset(name=config.settings.dataset_name)
recs = ds.to_df()
rec = recs[recs["tags"].map(lambda t: t.get("source") == "human_fail")].iloc[0]   # a loop-1 failure
print(ds.name)
{"inputs": rec["inputs"], "expectations": rec["expectations"], "tags": rec["tags"]}

## 4. Update: a person writes v2

The fix names the four policy files (OpenClaw's `read` can't list a folder) and deletes the
stale quick facts. Nothing else changes: the other three flaws are next loops' work. It is
registered as a new version; `production` still points at v1.

In [ ]:
print(instructions.diff(r["production_before"], r["candidate"]))

**Under the hood:** the candidate is `mlflow.genai.register_prompt(...)` with the tags `author` and `status=candidate`.
Registering never moves an alias, so production keeps running v1 until someone promotes. The tags
below are as of the end of the run; the gate added its verdict and run id:

In [ ]:
c = mlflow.genai.load_prompt(f"prompts:/{name}/{r['candidate']}")
print(f"v{c.version}: {c.commit_message}")
{k: v for k, v in c.tags.items() if k in ("author", "gate", "gate_run")}

## 5. Gate

Production and the candidate answer every dataset row fresh, scored by the same verifier. The
candidate ships only if every must-pass scorer is at 100%, it is not worse on train, it is not
worse on the holdout, and it is better somewhere.

In [ ]:
print(r["gate"]["outcome"], "-", r["gate"]["message"])
s.show(s.gate_table(1), pct=True)

**Under the hood:** the gate is one MLflow run with four nested runs: train and holdout, for production and for the
candidate. Each nested run is `mlflow.genai.evaluate(data=<dataset rows>, predict_fn=<the agent
pinned to one version>, scorers=[...])`, so both versions answer every row fresh, and those traces
are tagged `si_source = 'eval'`. The rules themselves are harness code (`selfimprove/gate.py`). The
parent run keeps the decision: a metric per scorer, `prompt_growth`, the tags `si.gate_passed` and
`si.gate_reasons`, and the artifacts `gate_table.txt` and `candidate.diff`:

In [ ]:
gate_run = mlflow.search_runs(
    experiment_ids=[experiment_id], output_format="list", order_by=["start_time DESC"],
    filter_string=f"tags.`si.job` = 'gate' AND tags.`si.candidate_version` = '{r['candidate']}'")[0]
print(gate_run.info.run_name, {k: v for k, v in gate_run.data.tags.items() if k.startswith("si.")})
children = mlflow.search_runs(experiment_ids=[experiment_id], output_format="list",
                              filter_string=f"tags.mlflow.parentRunId = '{gate_run.info.run_id}'")
print(sorted(ch.info.run_name for ch in children))
print(mlflow.artifacts.load_text(f"runs:/{gate_run.info.run_id}/gate_table.txt"))

## 6. Measure: the same traffic, after the promotion

A person promoted v2. The same eight questions went to production again and were reviewed again:

In [ ]:
s.show(s.before_after(1).astype({"failure_rate": float}), pct=False)

In [ ]:
after = s.loop_traces(1, "after")
s.show(s.answers(after))

In [ ]:
s.tree(after[0])

**Under the hood:** `selfimprove promote 2` is two alias moves, `mlflow.genai.set_prompt_alias(name, "previous", 1)`
and `set_prompt_alias(name, "production", 2)`, plus version tags that record who promoted, why, and
what it replaced. A rollback moves `production` back. The agent loads the alias on every call, so
neither needs a redeploy:

In [ ]:
v2 = mlflow.genai.load_prompt(f"prompts:/{name}/{r['candidate']}")
{k: v for k, v in v2.tags.items() if k in ("promoted_by", "promoted_because", "replaced_version", "replaced_failure_rate")}

**What a person did in this loop:** everything. Read traces, labeled them, turned failures into
tests, wrote the fix, read the gate's table, promoted. The next loop hands the store and update
steps to the harness. Open the MLflow UI (`Traces`, `Prompts`, `Datasets`, and the `gate` run) to
see the same evidence there.